In [6]:
import os
import json
import asyncio
import aiohttp
from PIL import Image
from io import BytesIO
from datasets import load_dataset
from torchvision import transforms
from itertools import islice

# ==============================
# CONFIG
# ==============================

SAVE_DIR = "laion_5m_64"
IMAGE_DIR = os.path.join(SAVE_DIR, "images")
CAPTION_FILE = os.path.join(SAVE_DIR, "captions.jsonl")

TARGET_IMAGES = 5_000_000
MAX_CONCURRENT = 512   # tune based on network
SIMILARITY_THRESHOLD = 0.28
IMAGE_SIZE = 1024



os.makedirs(IMAGE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "checkpoint.json")

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f)
    return {"dataset_position": 0, "saved_images": 0}

def save_checkpoint(dataset_position, saved_images):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({
            "dataset_position": dataset_position,
            "saved_images": saved_images
        }, f)

# ==============================
# TRANSFORM
# ==============================

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
])

# ==============================
# LOAD STREAMING DATASET
# ==============================

dataset = load_dataset(
    "laion/relaion400m",
    split="train",
    streaming=True
)

dataset = dataset.filter(
    lambda x: x["NSFW"] == "UNLIKELY"
    and x["similarity"] > SIMILARITY_THRESHOLD
    and len(x["caption"]) > 5
)

# ==============================
# ASYNC DOWNLOAD
# ==============================

semaphore = asyncio.Semaphore(MAX_CONCURRENT)

async def download_and_save(session, sample, index):
    async with semaphore:
        try:
            async with session.get(sample["url"], timeout=10) as response:
                if response.status != 200:
                    return None

                content = await response.read()
                image = Image.open(BytesIO(content)).convert("RGB")
                image = transform(image)

                filename = f"{index:010d}.jpg"
                filepath = os.path.join(IMAGE_DIR, filename)
                image.save(filepath, format="JPEG", quality=90)

                return {
                    "file": filename,
                    "caption": sample["caption"].strip()
                }

        except Exception:
            return None


async def main():
    checkpoint = load_checkpoint()
    dataset_position = checkpoint["dataset_position"]
    saved_images = checkpoint["saved_images"]

    print(f"Resuming from dataset position {dataset_position}")
    print(f"Already saved {saved_images} images")

    dataset_iter = islice(dataset, dataset_position, None)

    tasks = []
    samples_processed = dataset_position

    async with aiohttp.ClientSession() as session:
        for sample in dataset_iter:

            if saved_images >= TARGET_IMAGES:
                break

            task = asyncio.create_task(
                download_and_save(session, sample, saved_images)
            )
            tasks.append(task)

            samples_processed += 1

            if len(tasks) >= MAX_CONCURRENT:
                results = await asyncio.gather(*tasks)
                tasks = []

                with open(CAPTION_FILE, "a") as f:
                    for r in results:
                        if r:
                            f.write(json.dumps(r) + "\n")
                            saved_images += 1

                save_checkpoint(samples_processed, saved_images)

                print(f"Saved: {saved_images} | Dataset read: {samples_processed}")

        # Final batch
        results = await asyncio.gather(*tasks)
        with open(CAPTION_FILE, "a") as f:
            for r in results:
                if r:
                    f.write(json.dumps(r) + "\n")
                    saved_images += 1

        save_checkpoint(samples_processed, saved_images)

    print("Finished.")

await main()

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

Resuming from dataset position 0
Already saved 0 images
Saved: 326 | Dataset read: 512
Saved: 637 | Dataset read: 1024
Saved: 967 | Dataset read: 1536
Saved: 1305 | Dataset read: 2048
Saved: 1625 | Dataset read: 2560
Saved: 1764 | Dataset read: 3072
Saved: 2048 | Dataset read: 3584
Saved: 2265 | Dataset read: 4096
Saved: 2605 | Dataset read: 4608
Saved: 2835 | Dataset read: 5120
Saved: 3129 | Dataset read: 5632
Saved: 3476 | Dataset read: 6144
Saved: 3816 | Dataset read: 6656
Saved: 4163 | Dataset read: 7168
Saved: 4501 | Dataset read: 7680
Saved: 4861 | Dataset read: 8192
Saved: 5185 | Dataset read: 8704
Saved: 5498 | Dataset read: 9216
Saved: 5779 | Dataset read: 9728
Saved: 6102 | Dataset read: 10240
Saved: 6458 | Dataset read: 10752
Saved: 6761 | Dataset read: 11264
Saved: 7047 | Dataset read: 11776
Saved: 7391 | Dataset read: 12288
Saved: 7720 | Dataset read: 12800
Saved: 8025 | Dataset read: 13312
Saved: 8121 | Dataset read: 13824
Saved: 8307 | Dataset read: 14336
Saved: 8465 | D

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


Saved: 201721 | Dataset read: 355840
Saved: 202060 | Dataset read: 356352
Saved: 202401 | Dataset read: 356864
Saved: 202655 | Dataset read: 357376
Saved: 202839 | Dataset read: 357888
Saved: 203199 | Dataset read: 358400
Saved: 203553 | Dataset read: 358912
Saved: 203898 | Dataset read: 359424
Saved: 204229 | Dataset read: 359936
Saved: 204584 | Dataset read: 360448
Saved: 204911 | Dataset read: 360960
Saved: 205257 | Dataset read: 361472
Saved: 205602 | Dataset read: 361984
Saved: 205933 | Dataset read: 362496
Saved: 206092 | Dataset read: 363008
Saved: 206263 | Dataset read: 363520
Saved: 206421 | Dataset read: 364032
Saved: 206753 | Dataset read: 364544
Saved: 207103 | Dataset read: 365056
Saved: 207436 | Dataset read: 365568
Saved: 207773 | Dataset read: 366080
Saved: 208135 | Dataset read: 366592
Saved: 208473 | Dataset read: 367104
Saved: 208826 | Dataset read: 367616
Saved: 209162 | Dataset read: 368128
Saved: 209450 | Dataset read: 368640
Saved: 209791 | Dataset read: 369152
S

CancelledError: 